# 🛡️ AI Scam Hunter — Detect Suspicious Messages

An end-to-end, reproducible pipeline that classifies **fictional** emails/SMS as *suspicious* or *legitimate*, explains the warning signs, and produces a **structured JSON risk report**.

**What you will practise**

| Learning objective | Where |
|---|---|
| Fine-tuning vs. RAG | §1 (decision guide) and §7 (FAISS comparison) |
| Instruction tuning + structured outputs | §5 |
| LoRA / PEFT | §6 |
| Dataset quality: labels, dedup, imbalance, leakage | §3 |
| Evaluation-first: TF-IDF + LR baseline → fine-tuned model | §4, §6, §8 |

> **Safety note.** All data is synthetic/fictional (`.example` domains, `555` phone numbers). This tool is a screening aid: it can **never** guarantee a message is safe. The lowest-risk verdict is `no_strong_indicators`, and a low model score cannot override strong rule-based warning signs (see §9).

**Project layout**
```
ai_scam_hunter/
├── AI_Scam_Hunter.ipynb      <- this notebook (the story)
├── scam_hunter/              <- reusable modules (the engine)
│   ├── config.py  data_gen.py  data_quality.py  baseline.py  evaluate.py
│   ├── instruction_data.py  finetune.py  finetune_causal.py  rag_demo.py  report.py  stress_set.py
├── data/  artifacts/
└── requirements.txt
```

In [ ]:
import sys, json, os, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))          # the notebook lives in the project root
warnings.filterwarnings("ignore")

import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_colwidth", 110)

from scam_hunter.config import *
from scam_hunter.data_gen import generate_dataset
from scam_hunter.data_quality import (remove_duplicates, assign_splits, leakage_report, class_balance,
                                      class_weights, audit_labels)
from scam_hunter.baseline import majority_baseline, run_baseline, top_features
from scam_hunter.evaluate import compute_metrics, plot_confusion, comparison_table, choose_threshold, error_analysis
from scam_hunter.stress_set import load_stress_set
from scam_hunter.instruction_data import to_instruction_records, format_prompt, save_jsonl, INSTRUCTION
from scam_hunter.report import build_report, validate_report

np.random.seed(SEED)
print("Project root:", PROJECT_ROOT)

## 1. Fine-tuning vs. RAG vs. prompting — which tool for which job?

| | **Prompting** | **RAG** (retrieval) | **Fine-tuning** (e.g. LoRA) |
|---|---|---|---|
| Changes the model's weights? | No | No | Yes (a few %, with LoRA) |
| Best for | Quick prototypes | **Knowledge** that changes or must be cited | **Behaviour/format/style**, a stable task, consistent structured output |
| Handles new facts without retraining? | Limited by context | ✅ update the index | ❌ retrain |
| Latency / cost at inference | Long prompts | Extra retrieval step | Cheapest per call once trained |
| Typical failure | Format drift | Bad retrieval → bad answer | Overfits / forgets; needs quality labels |

**Rule of thumb:** *RAG for what the model should **know**, fine-tuning for how the model should **behave**.*

For scam detection both are plausible:
- **Fine-tuning** teaches the *decision* (is this a scam?) and the *output format* (JSON report).
- **RAG** is useful for *grounding* explanations in scam-pattern notes, and as a no-training **k-NN classifier** over labelled examples — we compare it in §7.

The **evaluation-first mindset**: before any fancy model, build a cheap baseline and a trustworthy test set. A fine-tuned model is only "better" if the numbers say so.

## 2. Data — a fictional, license-clean dataset

In [ ]:
raw = generate_dataset(seed=SEED)
raw.to_csv(RAW_CSV, index=False)
print(f"{len(raw)} raw messages")
raw.sample(6, random_state=1)[["channel", "category", "label", "text"]]

Every message is built from a *category* (e.g. `bank_phishing`) and a *body template* (`group`). Openers, closers and subjects are partly shared between scam and legitimate messages, so surface formatting alone doesn't give the label away. The dataset deliberately includes **hard negatives** (legitimate messages with links, urgency, "verify", "locked", "claim") and **subtle scams** (no links, little urgency).

Duplicates were injected on purpose so the next section has real work to do.

## 3. Dataset quality

Models inherit the flaws of their data. Four checks, in the order you would do them on a real project.

### 3.1 Class imbalance

In [ ]:
counts = raw["label"].map(LABELS).value_counts()
print(counts.to_string(), f"\n\nSuspicious share: {raw['label'].mean():.1%}")
raw["category"].value_counts().sort_index().plot.barh(color=raw.groupby("category")["label"].first().sort_index().map({0: "#4C78A8", 1: "#E45756"}))
plt.title("Messages per category (red = suspicious)"); plt.tight_layout(); plt.show()
print("Inverse-frequency class weights:", {LABELS[k]: round(v, 2) for k, v in class_weights(raw["label"]).items()})

Real inboxes are far more imbalanced (scams are a small minority), so **accuracy is misleading**: a model that always says "legitimate" would look good. We therefore report **precision, recall and F1 for the *suspicious* class**, use `class_weight="balanced"` / a weighted loss, and tune the decision threshold for recall on *validation* data.

### 3.2 Deduplication

In [ ]:
clean, dedup_info = remove_duplicates(raw)
print(dedup_info)

Exact and near-exact (case/whitespace/punctuation) duplicates are removed. Why it matters: duplicates that land in both train and test make a model look better than it is. We also check for **label conflicts** (identical text, different labels) — `label_conflicts` should be 0.

### 3.3 Leakage prevention: split by *template group*, not by row

In [ ]:
data = assign_splits(clean, seed=SEED)
data.to_csv(CLEAN_CSV, index=False)
train, val, test = (data[data.split == s].reset_index(drop=True) for s in ("train", "val", "test"))
display(class_balance(data))
print(leakage_report(data))

A random row-level split would put different *variants of the same template* in train and test, and the model would simply memorise templates. Instead, each category's body templates are assigned **wholesale** to one split, so **the test set contains templates the model has never seen**. The leakage report confirms zero exact overlap, zero group overlap, and no near-duplicates (TF-IDF cosine ≥ 0.9) between train and test.

Other leakage rules we follow: tune thresholds on *validation* only; never touch the test set or the stress set (§4) for model selection.

### 3.4 Label audit — finding mislabelled examples

In [ ]:
# Demo: corrupt 12 training labels, then see whether a cross-validated model flags them.
rng = np.random.default_rng(SEED)
noisy = train.copy()
flip_idx = rng.choice(len(noisy), size=12, replace=False)
noisy.loc[flip_idx, "label"] = 1 - noisy.loc[flip_idx, "label"]

flagged = audit_labels(noisy["text"], noisy["label"], flag_below=0.3)
caught = len(set(flagged.index) & set(flip_idx))
print(f"Flagged {len(flagged)} rows for review; {caught}/12 injected label flips were among them.")
flagged.head(5)

The audit trains on other folds and flags rows where the model strongly disagrees with the given label. Flagged rows go to a **human reviewer** — they are candidates, not verdicts. (The real training data stays untouched; this was a demo on a corrupted copy.)

## 4. Evaluation-first: the TF-IDF + Logistic Regression baseline

Before touching a transformer we need (a) a floor to beat and (b) a test we trust.

In [ ]:
results_test, results_stress = {}, {}
scores_test, preds_test, scores_stress, preds_stress = {}, {}, {}, {}
stress = load_stress_set()

results_test["Majority class"] = majority_baseline(train, test)

pipe, p_test, s_test, m = run_baseline(train, test)
results_test["TF-IDF + LogReg"] = m
scores_test["TF-IDF + LogReg"], preds_test["TF-IDF + LogReg"] = s_test, p_test

s_st = pipe.predict_proba(stress["text"])[:, 1]
scores_stress["TF-IDF + LogReg"] = s_st
preds_stress["TF-IDF + LogReg"] = (s_st >= 0.5).astype(int)
results_stress["TF-IDF + LogReg"] = compute_metrics(stress["label"], preds_stress["TF-IDF + LogReg"], s_st)

print("Held-out template test set"); display(comparison_table({k: v for k, v in results_test.items()}))
print("Hand-written stress set (novel wording, typos, benign messages with scammy words)")
display(comparison_table(results_stress))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(8, 3.4))
plot_confusion(test["label"], preds_test["TF-IDF + LogReg"], "Baseline — test", ax[0])
plot_confusion(stress["label"], preds_stress["TF-IDF + LogReg"], "Baseline — stress", ax[1])
plt.tight_layout(); plt.show()

pos, neg = top_features(pipe, 10)
print("Strongest 'suspicious' features:", pos)
print("Strongest 'legitimate' features:", neg)

**Reading the result honestly.** Template-generated data is easy, so the baseline scores near-perfectly on `test`. The **stress set** is the more informative number: novel phrasing, leetspeak and benign messages that *contain scam words* ("Urgent", "refund", "Winner", "Final call") expose a keyword-matching model. The errors below show it:

In [ ]:
error_analysis(stress, preds_stress["TF-IDF + LogReg"], n=12)

In [ ]:
# Tune the decision threshold for recall on VALIDATION data only (missing a scam costs more than a false alarm).
val_scores = pipe.predict_proba(val["text"])[:, 1]
thr = choose_threshold(val["label"], val_scores, min_recall=0.95)
print(f"Validation-tuned threshold for >=95% recall: {thr}  (default 0.5)")

## 5. Instruction tuning: instruction → structured JSON

**Instruction tuning** trains a model on *(instruction, input, output)* examples so it follows a task description and produces a **fixed output format**. We re-express the labelled messages as such records. The target is a JSON risk report — the model is rewarded for emitting valid, schema-conforming JSON, not free text.

In [ ]:
records = to_instruction_records(train.head(200))
print(format_prompt(records[0]) + records[0]["output"])

In [ ]:
for name, df_ in (("train", train), ("val", val), ("test", test)):
    save_jsonl(to_instruction_records(df_), DATA_DIR / f"instructions_{name}.jsonl")
print("Saved:", sorted(p.name for p in DATA_DIR.glob("instructions_*.jsonl")))

Key details (implemented in `scam_hunter/finetune_causal.py`):
- **Same prompt template for training and inference** (`format_prompt`) — mismatches are a classic silent bug.
- **Loss masking**: labels for the prompt tokens are `-100`, so the model is only trained to produce the *response*.
- **Validate outputs**: `validate_report()` rejects anything that is not valid JSON with the expected keys and verdict values.

Stage 3 (optional, GPU recommended): `train_instruction_model(records, "artifacts/sft")` LoRA-tunes a small causal LM (`Qwen/Qwen2.5-0.5B-Instruct`) on these records. The core project does not depend on it: §9 produces valid reports from the classifier + transparent rules.

## 6. LoRA / PEFT fine-tuning of a pretrained transformer

**Idea.** Instead of updating a pretrained weight matrix `W` (size d×d), learn a low-rank update `ΔW = B·A` where `A` is r×d and `B` is d×r with a small rank (`r = 8`). Only `A` and `B` (plus the new classification head) are trained: usually **~1–3 % of the parameters**, a tiny adapter file, and far less memory — while keeping the pretrained knowledge frozen.

We use `distilbert-base-uncased` with LoRA on the attention query/value projections. Training is **class-weighted** to counter imbalance, and model selection uses validation F1.

> Requires `torch`, `transformers`, `datasets`, `peft`, `accelerate` and a one-off model download from the Hugging Face Hub. If they aren't available, this section is skipped and the rest of the notebook still runs.

In [ ]:
RUN_FINETUNE = True          # set False to skip training
EPOCHS = 4

try:
    import torch, transformers, peft, datasets
    from scam_hunter.finetune import train_classifier, predict_proba, save_adapter
    HAS_TRANSFORMERS = True
except ImportError as e:
    HAS_TRANSFORMERS = False
    print("Transformer stack not installed -> skipping LoRA section.", e)

lora_model = None
if HAS_TRANSFORMERS and RUN_FINETUNE:
    trainer, tok, counts = train_classifier(train, val, str(ARTIFACTS_DIR / "lora_run"), use_lora=True, epochs=EPOCHS)
    lora_model = trainer.model
    print("Parameter counts:", counts)

In [ ]:
if lora_model is not None:
    name = "DistilBERT + LoRA"
    s_test = predict_proba(lora_model, tok, test["text"]); scores_test[name] = s_test
    preds_test[name] = (s_test >= 0.5).astype(int)
    results_test[name] = compute_metrics(test["label"], preds_test[name], s_test)

    s_st = predict_proba(lora_model, tok, stress["text"]); scores_stress[name] = s_st
    preds_stress[name] = (s_st >= 0.5).astype(int)
    results_stress[name] = compute_metrics(stress["label"], preds_stress[name], s_st)

    save_adapter(lora_model, tok, str(ARTIFACTS_DIR / "scam_hunter_lora"))
    print("Saved adapter to", ARTIFACTS_DIR / "scam_hunter_lora")
else:
    print("LoRA model not trained in this run.")

**Optional experiment — LoRA vs. full fine-tuning.** Re-run `train_classifier(..., use_lora=False, lr=3e-5)` to compare trainable-parameter counts, training time and F1. Typically LoRA gets close to full fine-tuning at a fraction of the trainable parameters.

## 7. RAG comparison with FAISS

Here "RAG" means **retrieval instead of training**: embed every labelled training message, store the vectors in a **FAISS** index, and classify a new message by a similarity-weighted vote of its *k* nearest labelled neighbours. No weights are updated. We also retrieve scam-pattern notes to *ground an explanation*.

Embeddings use `sentence-transformers/all-MiniLM-L6-v2` when it can be downloaded, otherwise an offline TF-IDF + SVD fallback (the `backend` printed below says which one was used).

In [ ]:
import faiss
from scam_hunter.rag_demo import RagClassifier, retrieve_guidance

rag = RagClassifier(k=5).fit(train)
print("Embedding backend:", rag.embedder.backend, "| FAISS vectors:", rag.index.ntotal)

name = "RAG (FAISS k-NN)"
s_test = rag.predict_proba(test["text"]); scores_test[name] = s_test
preds_test[name] = (s_test >= 0.5).astype(int)
results_test[name] = compute_metrics(test["label"], preds_test[name], s_test)
s_st = rag.predict_proba(stress["text"]); scores_stress[name] = s_st
preds_stress[name] = (s_st >= 0.5).astype(int)
results_stress[name] = compute_metrics(stress["label"], preds_stress[name], s_st)

msg = stress["text"].iloc[1]
print("\nQuery:", msg)
for sim, lab, txt in rag.neighbours(msg, k=3):
    print(f"  sim={sim:.2f}  label={LABELS[lab]:<10}  {txt[:90]!r}")
print("\nRetrieved guidance:", [g["title"] for g in retrieve_guidance(msg, k=2)])

**Takeaway.** Retrieval is only as good as the labelled neighbours and the embedding. It shines when the knowledge base changes often (add a new scam example → instantly searchable, no retraining) and gives *traceable evidence*; it struggles when a new scam is phrased unlike anything stored. Fine-tuning generalises across wording but needs a retrain to learn new patterns. In production the two are complementary.

## 8. Model comparison

In [ ]:
print("=== Held-out template test set ==="); display(comparison_table({k: v for k, v in results_test.items() if k != "Majority class"}))
print("=== Hand-written stress set ===");    display(comparison_table(results_stress))

models = list(results_stress)
fig, axes = plt.subplots(1, len(models), figsize=(3.8 * len(models), 3.4), squeeze=False)
for ax, name in zip(axes[0], models):
    plot_confusion(stress["label"], preds_stress[name], name, ax)
plt.suptitle("Stress-set confusion matrices"); plt.tight_layout(); plt.show()

**How to read this table.**
- Prefer **recall** on *suspicious* when a missed scam is costly, and watch **precision** so users aren't flooded with false alarms.
- Compare models on the **stress set**, not just the easy test set. A model that wins only where the baseline already scores ~1.0 hasn't proven anything.
- With 40 stress messages, one example moves F1 by several points: treat differences as *indicative*, and grow the stress set (or add real, licensed data) before drawing strong conclusions.
- Report the baseline *and* the fine-tuned model together — "measurable improvement" means a number that moved on data neither model was tuned on.

## 9. Structured JSON risk reports (with guardrails)

`build_report()` combines a model probability with **transparent rule-based evidence** (the exact text that triggered each warning sign).

Guardrails:
1. There is **no "safe" verdict**; the lowest-risk label is `no_strong_indicators`.
2. If the model says "low risk" but ≥ 2 *strong* rule signals fire (credential request, payment request, remote access, secrecy), the verdict is raised to `uncertain`.
3. Probabilities in the middle band map to `uncertain`, so the system can abstain.
4. Every report carries a disclaimer and a recommended action.

In [ ]:
best = "DistilBERT + LoRA" if "DistilBERT + LoRA" in scores_stress else "TF-IDF + LogReg"
print("Reports generated with:", best)

demo_idx = [0, 1, 20, 25, 31]   # a mix of scams and legitimate messages
reports = []
for i in demo_idx:
    r = build_report(stress["text"].iloc[i], float(scores_stress[best][i]))
    validate_report(r)
    reports.append({"message": stress["text"].iloc[i], "true_label": LABELS[int(stress["label"].iloc[i])], "report": r})

print(json.dumps(reports[0], indent=2))
pd.DataFrame([{"msg": x["message"][:60], "truth": x["true_label"], "verdict": x["report"]["verdict"],
               "risk": x["report"]["risk_score"], "signs": [s["type"] for s in x["report"]["warning_signs"]]} for x in reports])

In [ ]:
# Guardrail check: no report may ever claim a message is "safe".
all_reports = [build_report(t, float(p)) for t, p in zip(stress["text"], scores_stress[best])]
for r in all_reports: validate_report(r)
print(pd.Series([r["verdict"] for r in all_reports]).value_counts().to_string())
assert not any("safe" == r["verdict"] for r in all_reports)

(ARTIFACTS_DIR / "sample_reports.json").write_text(json.dumps(reports, indent=2))
print("Saved artifacts/sample_reports.json")

## 10. Limitations, ethics, and next steps

**Limitations**
- The data is synthetic and template-based; real scams evolve, are multilingual and adversarial. Scores here are *not* a prediction of real-world accuracy.
- The stress set is tiny (40 messages). Grow it and add licensed/real data before trusting small differences.
- Rule-based signals are heuristics and will miss novel phrasing.
- A classifier can be evaded (leetspeak, images, new wording). Treat output as **decision support**, never a guarantee of safety.

**Responsible use:** keep humans in the loop, never present a low score as "safe", don't process real private messages without consent, and log/review errors.

**Next steps (only after the core is solid)**
1. Add more/real licensed data and a larger, versioned stress set; track experiments.
2. Try char n-gram TF-IDF as a stronger classical baseline (does the LoRA gain survive?).
3. Calibrate probabilities (Platt/isotonic) before using the `uncertain` band operationally.
4. Run the optional instruction-tuned generator (`finetune_causal.py`) and measure JSON validity rate.
5. Expose `build_report` via **FastAPI** (`POST /analyze`), then a small UI.